# SALM Fine-tuning 15 · Is this voice real? Anti-spoofing (ASVspoof 2019 LA)

> **Task** utterance → bona fide or spoof (TTS / voice conversion) · **Data** ASVspoof 2019 Logical Access
> **Metric** EER ↓ · **Result** 30.73% → **0.57%** EER (−98.1%) · **Time** ~2 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: deepfakes, 2019 edition

ASVspoof 2019 LA pairs real recordings with speech from 19 different TTS and voice-conversion systems. The
**evaluation set's attacks (A07-A19) never appear in training** (A01-A06), so a detector has to catch fakes it has
never met.

Two things about the base model:

1. Its **answers** say "bonafide" to nearly everything: 11% accuracy on an eval set that is 90% spoof.
2. Its **log-probabilities** know better. Rank clips by log P("spoof") − log P("bonafide") as the answer
   and the equal error rate (EER, where false alarms equal misses) is ~31%, better than a coin flip.

So the metric is EER on that log-odds score, not the accuracy of the generated word. The goal is a detector that
generalizes to unseen attacks.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data: a manifest sorted by class

The parquet files are sorted: all bona fide clips first, then all spoofs. Unshuffled, a 25k-row manifest streams
through Lhotse's 10k-row shuffle buffer as "real, real, …, fake, fake, …" (see tutorial 11 for what that does to a
model). The baseline cell below shuffles it on disk.

In [ ]:
d = M / "asvspoof"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py asvspoof", log="prep_spoof.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
import collections
for s in ("train", "dev", "test"):
    rows = [json.loads(l) for l in open(d / f"{s}.json")]
    print(f"{s:5s} {len(rows):6d} clips  {dict(collections.Counter(r['text'] for r in rows))}  attacks: {sorted({r['system_id'] for r in rows})}")

## 2. Baseline: scoring with log-odds

`vllm_task_eval.py --score-labels bonafide spoof` appends each label to the prompt, reads the exact log-probability of
every label token from vLLM's `prompt_logprobs`, and stores `label_score = log P(spoof) − log P(bonafide)`. Exact
sequence scores never depend on a top-k cut-off, and a label that can't be scored is an error rather than a guess.

In [ ]:
TASK = f"--task eer --max-tokens 8 --labels {d}/labels.txt --score-labels bonafide spoof"
P2 = ("Listen for artifacts of speech synthesis or voice conversion. Is this recording bonafide (real human speech) or "
      "spoof (synthetic)? Answer with one word: bonafide or spoof.")
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt {q(P2)}")):
    for s in ("dev", "test"):
        base[p, s] = evaluate(CKPT, d / f"{s}.json", f"spoof_base_{p}_{s}", TASK, extra)
    print(f"{p}  dev EER {100 * base[p, 'dev']['score']:5.2f}%   eval EER {100 * base[p, 'test']['score']:5.2f}%   "
          f"(accuracy of the generated word: {100 * base[p, 'test']['accuracy']:.1f}%)")
BEST_P = min(("p1", "p2"), key=lambda p: base[p, "dev"]["score"])  # prompt chosen on dev
BASELINE = 100 * base[BEST_P, "test"]["score"]  # its test score: the baseline fine-tuning is compared with
ORACLE = 100 * min(base[p, "test"]["score"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}% EER")
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(P2)} {pairs}")
sh(f"{PYTHON} {FT}/shuffle_manifest.py {d}/train_ft.json")

## 3. Config and training (~40 min)

In [ ]:
CFG = WORK / "configs" / "spoof.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'spoof'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name spoof_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 40 --max-steps 400 "
   f"--batch-tokens 32000 --limit-train-batches 60 --limit-val-batches 50 --save-top-k -1")
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 240 300")

## 4. Select on dev (dev is too easy)

Dev uses the *same* attacks as training, so it's solved and says little about unseen attacks. Expected dev EER: 0.00%
at both step 240 and step 300 (a perfect ranking of all 3,000 clips). On a tie the earlier step wins. Select on dev
anyway, and let the eval set judge.

In [ ]:
dev = {}
for step in (240, 300):
    out = EXPORT / f"spoof_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'spoof'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_spoof_{step}.log")
    dev[step] = 100 * evaluate(out, d / "dev_ft.json", f"spoof_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev EER {dev[step]:.2f}%")
BEST = min(dev, key=dev.get)

## 5. Test on 13 unseen attacks, once

In [ ]:
MODEL = EXPORT / f"spoof_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "spoof_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "spoof_librispeech", "--task asr --normalizer english")
FT_EER = 100 * ft["score"]
print(f"eval EER: base {BASELINE:.2f}% → fine-tuned {FT_EER:.2f}% ({(BASELINE - FT_EER) / BASELINE:+.1%} relative reduction)")
print(f"balanced accuracy of the generated word {100 * ft['balanced_accuracy']:.2f}%   LibriSpeech WER {100 * guard['score']:.2f}%")
result("15 baseline eval EER", BASELINE, 30.73, 0.5)
result("15 fine-tuned eval EER", FT_EER, 0.57, 0.5)

Expected results:

| | eval EER | balanced accuracy (generated word) | LibriSpeech WER |
|---|---:|---:|---:|
| base, prompt chosen on dev (p2) | 30.73% | 50.61% | 1.54% |
| **fine-tuned** | **0.57%** | **99.29%** | 1.75% |

**−98.1% relative** EER on 71,237 clips and 13 attack types never seen in training. That's in the range of dedicated
anti-spoofing systems published for this benchmark (~0.5–1%). The prompt chosen on dev (p2: 28.10% vs 34.26%) is also
the test-oracle (30.73% vs 32.70%). The fine-tuned model's generated word agrees with its score: 99.29% balanced
accuracy. LibriSpeech WER moves from 1.54% to 1.75%: the encoder learns to hear synthesis artifacts without losing much
transcription quality.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="spoof")

## Takeaways

1. **Score detectors with log-odds, not generated words.** The base model's answers were useless; its probabilities
   weren't.
2. **Check for class-sorted data** before a 25k-row run, not after.
3. **Watch the side effects.** Training the encoder on a non-ASR task can cost general ASR. Check it every time.